# Amazon ML Challenge: reproducible Kaggle run

This notebook is the portable launcher for the production package in `code/business_entity_resolution`. It resets the temporary code checkout, records the Git commit and hardware, locates the TSV dataset, runs the complete pipeline, validates both challenge TSV files, and builds the final submission archive.

The two Kaggle GPUs are used as independent distributed workers for multilingual E5 encoding. XGBoost uses one GPU because its candidate-pair table is a single-node tabular workload.

## 1. Run controls

On Kaggle, select the **T4 x2** accelerator and turn Internet on for the first run. If Internet must remain off, attach the pinned E5 model snapshot as a Kaggle Dataset and set `BER_E5_MODEL_PATH`.

In [ ]:
import os, shutil, subprocess, sys, tempfile, time
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
RUN_FULL_PIPELINE = IS_KAGGLE
REPO_URL = os.environ.get('BER_REPO_URL', 'https://github.com/saltypal/ASR_AmazonML.git')
GIT_REF = os.environ.get('BER_GIT_REF', 'main')
TEAM_NAME = os.environ.get('BER_TEAM_NAME', 'team_name')
print({'is_kaggle': IS_KAGGLE, 'run_full_pipeline': RUN_FULL_PIPELINE, 'git_ref': GIT_REF})

## 2. Clean reset and exact code sync

Kaggle runtimes are temporary. This cell deliberately removes only `/kaggle/working/amazonml-src`, clones the repository again, checks out the requested ref, installs the pinned package, and records the resolved commit. This prevents stale notebook state from changing an experiment.

In [ ]:
def locate_local_repo(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'code' / 'business_entity_resolution' / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the AmazonML repository.')

if IS_KAGGLE:
    PROJECT_DIR = Path('/kaggle/working/amazonml-src')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(['git', 'checkout', GIT_REF], cwd=PROJECT_DIR, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-e',
                    str(PROJECT_DIR / 'code' / 'business_entity_resolution')], check=True)
else:
    PROJECT_DIR = locate_local_repo(Path.cwd().resolve())

PACKAGE_ROOT = PROJECT_DIR / 'code' / 'business_entity_resolution'
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print('Project:', PROJECT_DIR)
print('Commit:', GIT_COMMIT)

## 3. Hardware and reproducibility check

The full Kaggle configuration expects CUDA. E5 encoding launches one process per visible GPU and shards Parquet batches by rank. The random seed controls the entity-level split and hyperparameter candidates.

In [ ]:
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'python': sys.version.split()[0], 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count(), 'gpus': GPU_NAMES})
if RUN_FULL_PIPELINE and torch.cuda.device_count() < 2:
    raise RuntimeError('Select Kaggle T4 x2 before starting the full run.')

## 4. Locate the challenge data and pin the allowed multilingual model

The input remains TSV. The pipeline reads it in chunks and writes temporary Parquet partitions for efficient repeated access; CSV conversion is unnecessary. `multilingual-e5-small` optionally reranks a bounded set of cross-script candidates and is pinned to a reviewed MIT-licensed revision. It does not query any external business database.

In [ ]:
import yaml

def find_data_root() -> Path:
    candidates = []
    if IS_KAGGLE:
        candidates.extend(Path('/kaggle/input').rglob('train_source1.tsv'))
    candidates.append(PROJECT_DIR / 'dataset' / 'train' / 'train_source1.tsv')
    for source1_file in candidates:
        root = source1_file.parent.parent
        required = [root / 'train' / 'train_ground_truth.tsv', root / 'test' / 'test_source3.tsv']
        if all(path.is_file() for path in required):
            return root
    raise FileNotFoundError('Attach the challenge dataset with dataset/train and dataset/test folders.')

DATA_ROOT = find_data_root()
MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = 'fd1525a9fd15316a2d503bf26ab031a61d056e98'
E5_MODEL_PATH = os.environ.get('BER_E5_MODEL_PATH')
if RUN_FULL_PIPELINE and not E5_MODEL_PATH:
    attached = list(Path('/kaggle/input').rglob('modules.json')) if IS_KAGGLE else []
    attached = [path.parent for path in attached if (path.parent / 'config.json').is_file()]
    if attached:
        E5_MODEL_PATH = str(attached[0])
    else:
        from huggingface_hub import snapshot_download
        E5_MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION,
                                          local_dir='/kaggle/temp/multilingual-e5-small')

BASE_CONFIG = PACKAGE_ROOT / 'configs' / ('kaggle_t4x2.yaml' if IS_KAGGLE else 'smoke.yaml')
runtime_config = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
if E5_MODEL_PATH:
    runtime_config.setdefault('embeddings', {})['model_path'] = E5_MODEL_PATH
RUNTIME_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path(tempfile.mkdtemp(prefix='ber-notebook-'))
RUNTIME_CONFIG = RUNTIME_ROOT / 'ber_runtime_config.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding='utf-8')
WORK_DIR = Path('/kaggle/temp/ber-run') if IS_KAGGLE else RUNTIME_ROOT / 'work'
OUTPUT_DIR = Path('/kaggle/working/submission/output') if IS_KAGGLE else RUNTIME_ROOT / 'output'
print({'data_root': str(DATA_ROOT), 'config': str(RUNTIME_CONFIG), 'model_path': E5_MODEL_PATH})

## 5. Inspect, then execute the complete bounded run

The run has a hard 480 minute budget with 35 minutes reserved for inference, TSV creation, validation, and packaging. Hyperparameter search is capped by both trials and wall time. Validation and holdout splitting is by Source‑1 entity, so candidate pairs for one entity never leak across splits.

In [ ]:
pipeline_env = os.environ.copy()
pipeline_env['PYTHONPATH'] = str(PACKAGE_ROOT / 'src') + os.pathsep + pipeline_env.get('PYTHONPATH', '')
base_command = [sys.executable, '-m', 'ber.cli', '--config', str(RUNTIME_CONFIG),
                '--repo-root', str(PROJECT_DIR), '--data-root', str(DATA_ROOT),
                '--work-dir', str(WORK_DIR), '--output-dir', str(OUTPUT_DIR)]
subprocess.run([*base_command[:3], 'inspect', *base_command[3:]], cwd=PACKAGE_ROOT, env=pipeline_env, check=True)

if RUN_FULL_PIPELINE:
    started = time.monotonic()
    subprocess.run([*base_command[:3], 'run-all', *base_command[3:]], cwd=PACKAGE_ROOT, env=pipeline_env, check=True)
    print(f'Complete runtime: {(time.monotonic() - started) / 60:.1f} minutes')
else:
    print('Local notebook validation mode: inspect succeeded; full dataset run was skipped.')

## 6. Persist outputs and build the challenge archive

Kaggle only preserves `/kaggle/working` when you save a version. Large normalized partitions and embedding shards stay under `/kaggle/temp`; only the two final TSV files, code, methodology template, model, and run evidence are retained.

In [ ]:
if RUN_FULL_PIPELINE:
    package_root = Path('/kaggle/working/package')
    if package_root.exists():
        shutil.rmtree(package_root)
    shutil.copytree(OUTPUT_DIR, package_root / 'output')
    shutil.copytree(PACKAGE_ROOT, package_root / 'code' / 'business_entity_resolution',
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'artifacts'))
    shutil.copy2(PROJECT_DIR / 'Documentation_template.md', package_root / 'Documentation_template.md')
    evidence = Path('/kaggle/working/run_evidence')
    evidence.mkdir(parents=True, exist_ok=True)
    for path in [WORK_DIR / 'run_metadata.json', WORK_DIR / 'completed_run.json']:
        shutil.copy2(path, evidence / path.name)
    shutil.copytree(WORK_DIR / 'model', evidence / 'model', dirs_exist_ok=True)
    archive_base = Path('/kaggle/working') / f'{TEAM_NAME}_submission'
    archive_path = shutil.make_archive(str(archive_base), 'zip', package_root)
    for path in [OUTPUT_DIR / 'matching_results.tsv', OUTPUT_DIR / 'candidate_pairs.tsv', Path(archive_path)]:
        print(f'{path}: {path.stat().st_size / 2**20:.2f} MiB')
    print('Save a Kaggle notebook version now so these files persist.')
else:
    print('Packaging runs after the full Kaggle pipeline completes.')

## Interpretation and next experiment

Do not treat a successful run as evidence of 0.99 accuracy. Inspect `work/completed_run.json` for measured validation and untouched holdout macro F0.5, and inspect candidate recall in each candidate manifest. If candidate recall is weak, improve blocking first. If recall is strong but F0.5 is weak, improve pair features or threshold calibration. Submit only the validated `matching_results.tsv`.